# MNIST Clustering — Model Comparison

This notebook runs inference using all pre-trained models on the MNIST dataset and compares their clustering performance.

**Models**: Baseline GMM, VAE-GMM, Diffusion-VAE, Ours  
**Metrics**: Accuracy (Hungarian), NMI, ARI  
**Clusters**: K=10 (one per digit)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from notebook_utils import (
    run_all_models, build_metrics_table, plot_cluster_means_grid,
    plot_all_models_cluster_samples, compute_cluster_purity_table,
    save_results_cache, MODEL_DISPLAY_NAMES,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Data & Run All Models

We evaluate on the MNIST test set (10,000 images). For each model we:
1. Extract latent features (or pixel features for baselines)
2. Fit a GMM with K=10 components
3. Assign cluster labels via the fitted GMM

For Diffusion-VAE, we use denoised latents at t=5 (best performing timestep).

In [ ]:
MODEL_ORDER = ['baseline_gmm', 'vae_gmm', 'diffusion_vae', 'clast']

print('Running inference on all models...')
results, (test_loader, dataset_info) = run_all_models('mnist', MODEL_ORDER, device)
print(f'\nModels evaluated: {[results[m]["display_name"] for m in MODEL_ORDER if m in results]}')

# Cache cluster assignments for sample browsing
save_results_cache(results, 'mnist')

## 2. Performance Metrics

We compare clustering performance using three standard metrics:
- **Accuracy**: Hungarian algorithm-based optimal matching between clusters and ground-truth labels
- **NMI**: Normalized Mutual Information — measures shared information between predicted and true labels
- **ARI**: Adjusted Rand Index — measures pairwise agreement adjusted for chance

In [ ]:
metrics_df = build_metrics_table(results, supervised=True)
metrics_df

## 3. Cluster Means Comparison

Each row shows the decoded GMM cluster means for a model. For neural models (VAE-GMM, Diffusion-VAE, Ours), the means are decoded from latent space. For Baseline GMM, they are the raw pixel-space centroids.

In [ ]:
plot_cluster_means_grid(results, MODEL_ORDER, device, title='MNIST — Cluster Means')

## 4. Cluster Purity (Ours)

Per-cluster breakdown showing which digit dominates each cluster and the purity ratio.

In [ ]:
if 'clast' in results:
    r = results['clast']
    purity_df = compute_cluster_purity_table(
        r['labels'], r['cluster_labels'], r['num_clusters'], dataset_info['class_names']
    )
    display(purity_df.style.format({'Purity': '{:.2%}'}).background_gradient(
        subset=['Purity'], cmap='Greens', vmin=0, vmax=1
    ))

## 5. Cluster Samples

Change `CLUSTER_IDX` below and re-run the cell to browse samples from different clusters.

In [ ]:
CLUSTER_IDX = 0  # <-- Change this value (0-9) and re-run

plot_all_models_cluster_samples(results, MODEL_ORDER, CLUSTER_IDX, n_samples=8)

## Analysis

**Key observations:**

- **Baseline GMM** operates in pixel space (784D) and produces blurry centroids, achieving the lowest performance. The high dimensionality makes GMM fitting difficult.

- **VAE-GMM** learns a compact latent space where GMM clustering is more effective. Cluster means are recognizable digits, showing the VAE decoder generates coherent prototypes.

- **Diffusion-VAE** applies a latent denoiser on top of VAE-GMM. The denoised latent codes can improve cluster separation, though the improvement depends on the denoising timestep.

- **Ours** achieves the best clustering metrics by using manifold-aware EM with heat-kernel graph medoid selection. The cluster prototypes are the sharpest and most representative.